In [1]:
pip install sklearn-crfsuite

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 18.9 MB/s eta 0:00:00a 0:00:01
Note: you may need to restart the kernel to use updated packages.


In [2]:
import re

In [3]:
import pandas as pd

file_path = "/kaggle/input/datasets/karimmahmoud09/pii-safety-dataset/transformed_train.parquet"
df = pd.read_parquet(file_path)

df.head()

,masked_text,unmasked_text,token_entity_labels,tokenised_unmasked_text
0,[PREFIX_1] [FIRSTNAME_1] [MIDDLENAME_1] [LASTN...,"Mr. Adolphus Reagan Ziemann, as a Central Prin...","[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, ...","[mr, ., adolph, ##us, reagan, z, ##ie, ##mann,..."
1,"Hello [FIRSTNAME_1], would you please investig...","Hello Hannah, would you please investigate the...","[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, ...","[hello, hannah, ,, would, you, please, investi..."
2,We also request a review of our policies with ...,We also request a review of our policies with ...,"[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, ...","[we, also, request, a, review, of, our, polici..."
3,"Dear [FIRSTNAME_1], a company-wide presentatio...","Dear Devan, a company-wide presentation is req...","[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, ...","[dear, dev, ##an, ,, a, company, -, wide, pres..."
4,Can we also have a session on how to manage st...,Can we also have a session on how to manage st...,"[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, ...","[can, we, also, have, a, session, on, how, to,..."


In [4]:
sentences = []
s=set()
for _, row in df.iterrows():
    token_entity_labels = row["token_entity_labels"]
    tokenised_unmasked_text = row["tokenised_unmasked_text"]
    for val in token_entity_labels:
        s.add(val)
    sentence = list(zip(tokenised_unmasked_text, token_entity_labels))
    sentences.append(sentence)

In [5]:
def word2features(sent, i):
    word = sent[i][0]

    features = {
        "bias": 1.0,
        "word.lower()": word.lower(),
        "word[-3:]": word[-3:],
        "word[-2:]": word[-2:],
        "word.isupper()": word.isupper(),
        "word.istitle()": word.istitle(),
        "word.isdigit()": word.isdigit(),
    }

    # Previous word features
    if i > 0:
        prev_word = sent[i - 1][0]
        features.update({
            "-1:word.lower()": prev_word.lower(),
            "-1:word.istitle()": prev_word.istitle(),
            "-1:word.isupper()": prev_word.isupper(),
        })
    else:
        features["BOS"] = True  # Beginning of sentence

    # Next word features
    if i < len(sent) - 1:
        next_word = sent[i + 1][0]
        features.update({
            "+1:word.lower()": next_word.lower(),
            "+1:word.istitle()": next_word.istitle(),
            "+1:word.isupper()": next_word.isupper(),
        })
    else:
        features["EOS"] = True  

    return features

In [6]:
def extract_features(sentences):
    X = []
    y = []

    for sent in sentences:
        X.append([word2features(sent, i) for i in range(len(sent))])
        y.append([label for (_, label) in sent])

    return X, y

In [7]:
from sklearn.model_selection import train_test_split

X, y = extract_features(sentences)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [8]:
import sklearn_crfsuite
from sklearn_crfsuite import metrics

crf = sklearn_crfsuite.CRF(
    algorithm='lbfgs',
    c1=0.1,   # L1 regularization
    c2=0.1,   # L2 regularization
    max_iterations=100,
    all_possible_transitions=True
)

crf.fit(X_train, y_train)

CRF(algorithm='lbfgs', all_possible_transitions=True, c1=0.1, c2=0.1,
    max_iterations=100)

In [9]:
y_pred = crf.predict(X_test)

print(metrics.flat_classification_report(
    y_test, y_pred, digits=3
))

                    precision    recall  f1-score   support

     B-ACCOUNTNAME      0.981     0.995     0.988       206
   B-ACCOUNTNUMBER      0.732     0.481     0.580       210
B-CREDITCARDNUMBER      0.759     0.629     0.688       205
           B-EMAIL      0.924     0.927     0.925       300
            B-IPV4      0.758     0.723     0.740       242
            B-IPV6      0.781     0.673     0.723       196
             B-MAC      0.989     0.995     0.992       189
        B-PASSWORD      0.763     0.478     0.588       209
    B-PHONE_NUMBER      0.956     0.895     0.925       219
             B-SSN      0.971     0.754     0.849       175
        B-USERNAME      0.800     0.530     0.638       279
     I-ACCOUNTNAME      0.981     0.997     0.989       355
   I-ACCOUNTNUMBER      0.620     0.485     0.544       769
I-CREDITCARDNUMBER      0.775     0.654     0.710      1755
           I-EMAIL      0.978     0.996     0.987      2506
            I-IPV4      0.751     0.723

# additional features

In [10]:
def word2features2(sent, i):
    word = sent[i][0]

    features = {
        "bias": 1.0,
        "word.lower()": word.lower(),
        "word[-3:]": word[-3:],
        "word[-2:]": word[-2:],
        "word.isupper()": word.isupper(),
        "word.istitle()": word.istitle(),
        "word.isdigit()": word.isdigit(),
        "word.has_dot()": "." in word,
        "word.has_digit()": any(c.isdigit() for c in word),
        "word.has_colon()": ":" in word,
        "word.is_hex()": bool(re.fullmatch(r"[0-9a-fA-F]+", word)),
    }

    # Previous word features
    if i > 0:
        prev_word = sent[i - 1][0]
        features.update({
            "-1:word.lower()": prev_word.lower(),
            "-1:word.istitle()": prev_word.istitle(),
            "-1:word.isupper()": prev_word.isupper(),
        })
    else:
        features["BOS"] = True  # Beginning of sentence

    # Next word features
    if i < len(sent) - 1:
        next_word = sent[i + 1][0]
        features.update({
            "+1:word.lower()": next_word.lower(),
            "+1:word.istitle()": next_word.istitle(),
            "+1:word.isupper()": next_word.isupper(),
        })
    else:
        features["EOS"] = True  

    return features

In [11]:
def extract_features2(sentences):
    X = []
    y = []

    for sent in sentences:
        X.append([word2features2(sent, i) for i in range(len(sent))])
        y.append([label for (_, label) in sent])

    return X, y

In [12]:
from sklearn.model_selection import train_test_split

X, y = extract_features2(sentences)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [13]:
import sklearn_crfsuite
from sklearn_crfsuite import metrics

crf = sklearn_crfsuite.CRF(
    algorithm='lbfgs',
    c1=0.1,   # L1 regularization
    c2=0.1,   # L2 regularization
    max_iterations=200,
    all_possible_transitions=True
)

crf.fit(X_train, y_train)

CRF(algorithm='lbfgs', all_possible_transitions=True, c1=0.1, c2=0.1,
    max_iterations=200)

In [14]:
y_pred = crf.predict(X_test)

print(metrics.flat_classification_report(
    y_test, y_pred, digits=3
))

                    precision    recall  f1-score   support

     B-ACCOUNTNAME      0.976     0.995     0.986       206
   B-ACCOUNTNUMBER      0.737     0.548     0.628       210
B-CREDITCARDNUMBER      0.751     0.649     0.696       205
           B-EMAIL      0.923     0.923     0.923       300
            B-IPV4      0.754     0.736     0.745       242
            B-IPV6      0.739     0.694     0.716       196
             B-MAC      1.000     1.000     1.000       189
        B-PASSWORD      0.752     0.507     0.606       209
    B-PHONE_NUMBER      0.966     0.900     0.931       219
             B-SSN      0.966     0.811     0.882       175
        B-USERNAME      0.809     0.563     0.664       279
     I-ACCOUNTNAME      0.975     0.997     0.986       355
   I-ACCOUNTNUMBER      0.670     0.545     0.601       769
I-CREDITCARDNUMBER      0.761     0.677     0.716      1755
           I-EMAIL      0.980     0.996     0.988      2506
            I-IPV4      0.749     0.738